# 🧪 Scanntech QA Validator — Etapa 01

> **Escopo da Etapa 01:** Vendas simples, cancelamentos, desconto/acréscimo por operador,
> multi-pagamento, item pesável e fechamento por POS.
> Sem promoção Scanntech — foco em estrutura, valores e integridade do cupom.

### Fluxo de execução

| # | Célula | O que faz |
|---|--------|-----------|
| 1 | Instalação | Instala `openpyxl`, `pandas`, `pdfplumber` |
| 2 | Repositório | Clona/atualiza `automacao_scann` do GitHub |
| 3 | Health-check | Verifica módulos e imprime versões |
| 4 | Upload Roteiro | Sobe o TEMPLATE xlsx da Etapa 01 |
| 5 | Upload Audit | Sobe o export Audit xlsx |
| 6 | Upload PDF | Sobe o PDF de cupons fiscais |
| 7 | ▶️ Validação | Executa os 10 checks por linha |
| 8 | Resultados | Exibe resumo, tabela e checks detalhados |
| 9 | Download | Baixa o xlsx preenchido + log JSON |
| 10 | Diagnóstico | Inspeciona um teste específico |
| 11 | Teste Unitário | Roda mocks para validar módulos sem arquivos reais |

---
## 📦 Célula 1 — Instalação de dependências

In [ ]:
!pip install -q pandas openpyxl pdfplumber

import importlib, sys
print('Versões instaladas:')
for lib in ['pandas', 'openpyxl', 'pdfplumber']:
    mod = importlib.import_module(lib)
    ver = getattr(mod, '__version__', 'ok')
    print(f'  ✅ {lib} {ver}')
print('\n✅ Todas as dependências instaladas!')

---
## 🗂️ Célula 2 — Clonar / atualizar repositório

In [ ]:
import os, sys

REPO_URL = 'https://github.com/Miked0/automacao_scann.git'
REPO_DIR = 'automacao_scann'

os.chdir('/content')

if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
    !git pull origin main
    print(f'✅ Repositório atualizado → {os.getcwd()}')
else:
    !git clone {REPO_URL}
    os.chdir(REPO_DIR)
    print(f'✅ Repositório clonado → {os.getcwd()}')

ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# Força reload dos módulos src/* para garantir versão mais recente
import importlib
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('src.'):
        importlib.reload(sys.modules[mod_name])

os.makedirs('input',  exist_ok=True)
os.makedirs('output', exist_ok=True)

MODULOS = [
    'src/__init__.py', 'src/file_loader.py', 'src/audit_parser.py',
    'src/coupon_pdf_parser.py', 'src/promo_engine.py', 'src/test_runner.py',
    'src/result_writer.py', 'src/audit_logger.py', 'scanntech_qa_validator.py',
]
print()
todos_ok = True
for m in MODULOS:
    ok = os.path.exists(m)
    if not ok: todos_ok = False
    print(f'  {"✅" if ok else "❌"} {m}')
print(f'\n📂 Diretório: {ROOT}')
print('✅ Todos os módulos encontrados!' if todos_ok else '⚠️  Módulos faltando!')

---
## 🩺 Célula 3 — Health-check dos módulos

In [ ]:
import importlib, traceback

IMPORTS = [
    ('src.file_loader',       'FileLoader'),
    ('src.audit_parser',      'AuditParser'),
    ('src.coupon_pdf_parser', 'CouponPDFParser'),
    ('src.promo_engine',      'PromoEngine'),
    ('src.test_runner',       'TestRunner'),
    ('src.result_writer',     'ResultWriter'),
    ('src.audit_logger',      'AuditLogger'),
]

print('🩺 Health-check dos módulos')
print('=' * 50)
erros = []
for modulo, classe in IMPORTS:
    try:
        mod = importlib.import_module(modulo)
        importlib.reload(mod)  # garante versão mais recente
        cls = getattr(mod, classe)
        print(f'  ✅ {modulo}.{classe}')
    except Exception as e:
        print(f'  ❌ {modulo}.{classe} → {e}')
        erros.append((modulo, traceback.format_exc()))

print()
if erros:
    print(f'⚠️  {len(erros)} erro(s):')
    for mod, tb in erros:
        print(f'\n--- {mod} ---')
        print(tb)
else:
    print('✅ Todos os módulos importados com sucesso!')

---
## 📋 Célula 4 — Upload do Roteiro (TEMPLATE xlsx)

> Suba o arquivo **TEMPLATE_COM_BIN_NOVO.xlsx** (ou equivalente da Etapa 01).

In [ ]:
from google.colab import files as colab_files
import shutil, openpyxl

print('=' * 55)
print('📋 Upload da planilha do roteiro (.xlsx)')
print('=' * 55)

uploaded = colab_files.upload()

INPUT_ROTEIRO = None
for fname in uploaded:
    dest = f'input/{fname}'
    with open(fname, 'wb') as fout:
        fout.write(uploaded[fname])
    shutil.copy(fname, dest)
    INPUT_ROTEIRO = dest

if INPUT_ROTEIRO:
    wb_prev = openpyxl.load_workbook(INPUT_ROTEIRO, data_only=True)
    ws_prev = wb_prev.active
    print(f'  ✅ Arquivo   : {INPUT_ROTEIRO}')
    print(f'  📄 Abas      : {wb_prev.sheetnames}')
    print(f'  📏 Dimensões : {ws_prev.max_row} linhas × {ws_prev.max_column} colunas')
    # Mostra quais células estão mescladas (diagnóstico)
    merges = list(ws_prev.merged_cells.ranges)
    if merges:
        print(f'  🔀 Merges encontrados: {len(merges)} (ex: {str(merges[0])})')
else:
    raise ValueError('❌ Nenhum arquivo enviado. Execute esta célula novamente.')

---
## 📊 Célula 5 — Upload do Export Audit (.xlsx)

> Suba o arquivo `export_tickets_audit_companyId-XXXXXX_*.xlsx`.

In [ ]:
import pandas as pd

print('=' * 55)
print('📊 Upload do export Audit (.xlsx)')
print('=' * 55)

uploaded_audit = colab_files.upload()

INPUT_AUDIT = None
for fname in uploaded_audit:
    dest = f'input/{fname}'
    with open(fname, 'wb') as fout:
        fout.write(uploaded_audit[fname])
    shutil.copy(fname, dest)
    INPUT_AUDIT = dest

if INPUT_AUDIT:
    try:
        _prev = pd.read_excel(INPUT_AUDIT, nrows=3)
        print(f'  ✅ Arquivo  : {INPUT_AUDIT}')
        print(f'  🔑 Colunas  : {list(_prev.columns)}')
        print(f'  📏 Preview  : {len(_prev)} linhas lidas (amostra)')
    except Exception as e:
        print(f'  ⚠️ Carregado mas erro na leitura: {e}')
else:
    raise ValueError('❌ Nenhum arquivo enviado. Execute esta célula novamente.')

---
## 📄 Célula 6 — Upload do PDF de cupons fiscais

> Suba o PDF mesclado com todos os cupons da Etapa 01.

In [ ]:
import pdfplumber

print('=' * 55)
print('📄 Upload do PDF de cupons fiscais')
print('=' * 55)

uploaded_pdf = colab_files.upload()

INPUT_PDF = None
for fname in uploaded_pdf:
    dest = f'input/{fname}'
    with open(fname, 'wb') as fout:
        fout.write(uploaded_pdf[fname])
    shutil.copy(fname, dest)
    INPUT_PDF = dest

if INPUT_PDF:
    with pdfplumber.open(INPUT_PDF) as _pdf:
        n_pags = len(_pdf.pages)
        primeira = (_pdf.pages[0].extract_text() or '')[:200].replace('\n', ' ')
    print(f'  ✅ Arquivo  : {INPUT_PDF}')
    print(f'  📄 Páginas  : {n_pags}')
    print(f'  🔍 Preview  : {primeira}...')
else:
    raise ValueError('❌ Nenhum arquivo enviado. Execute esta célula novamente.')

---
## ▶️ Célula 7 — Executar validação da Etapa 01

> Usa os 7 módulos do projeto para executar os **10 checks** em cada linha do roteiro.
> Gera `output/etapa01_resultado.xlsx` e `output/qa_audit_log.json`.

In [ ]:
import logging, sys, importlib
from openpyxl.utils import get_column_letter
from openpyxl.styles import PatternFill

# ── Força reload dos módulos src para garantir versão do GitHub ──
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('src.'):
        try: importlib.reload(sys.modules[mod_name])
        except Exception: pass

# Reconfigura logging
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler('output/qa_validation.log', encoding='utf-8'),
    ],
)

from src.file_loader       import FileLoader
from src.audit_parser      import AuditParser
from src.coupon_pdf_parser import CouponPDFParser
from src.promo_engine      import PromoEngine
from src.test_runner       import TestRunner
from src.result_writer     import ResultWriter, KEYWORDS_HEADER
from src.audit_logger      import AuditLogger

OUTPUT_XLSX = 'output/etapa01_resultado.xlsx'
OUTPUT_LOG  = 'output/qa_audit_log.json'

print('🚀 Scanntech QA Validator — Etapa 01')
print('=' * 55)

# ── 1. FileLoader ────────────────────────────────────────
print('[M1] FileLoader: carregando artefatos...')
loader    = FileLoader(roteiro_path=INPUT_ROTEIRO, audit_path=INPUT_AUDIT, pdf_path=INPUT_PDF)
artefatos = loader.load_all()
wb        = artefatos['workbook']
audit_df  = artefatos['audit_df']
pdf_pages = artefatos['pdf_pages']
print(f'     Audit: {len(audit_df)} linhas | PDF: {len(pdf_pages)} páginas')

# ── 2. AuditParser ───────────────────────────────────────
print('[M2] AuditParser: indexando cupons...')
audit_parser = AuditParser(audit_df)
print(f'     {len(audit_parser.all_numeros())} cupons indexados')
print(f'     Exemplos: {audit_parser.all_numeros()[:5]}')

# ── 3. CouponPDFParser ───────────────────────────────────
print('[M3] CouponPDFParser: extraindo cupons do PDF...')
pdf_parser = CouponPDFParser(pdf_pages)
print(f'     {len(pdf_parser.coupons)} cupons extraídos do PDF')

# ── 4. PromoEngine ───────────────────────────────────────
promo_engine = PromoEngine()
print('[M4] PromoEngine: pronto')

# ── 5. Leitura do roteiro (com suporte a MergedCell) ─────
print('[M5] TestRunner: lendo roteiro...')
ws = wb.active

def _detect_header_merged(ws, keywords):
    # Varre células normais
    for row in ws.iter_rows():
        for cell in row:
            if cell.value and any(kw in str(cell.value).lower() for kw in keywords):
                return cell.row
    # Fallback: varre âncoras de merges
    for mr in ws.merged_cells.ranges:
        anchor = ws.cell(row=mr.min_row, column=mr.min_col)
        if anchor.value and any(kw in str(anchor.value).lower() for kw in keywords):
            return mr.min_row
    return 1

def _resolve_merged_headers(ws, header_row):
    # Mapa col_index -> nome, resolvendo merges
    col_map = {}
    for cell in ws[header_row]:
        if cell.value is not None:
            col_map[cell.column] = str(cell.value).strip().lower()
    # Propaga valor da âncora para colunas cobertas pelo merge
    for mr in ws.merged_cells.ranges:
        if mr.min_row <= header_row <= mr.max_row:
            anchor_val = ws.cell(row=mr.min_row, column=mr.min_col).value
            if anchor_val is None: continue
            anchor_str = str(anchor_val).strip().lower()
            for col_idx in range(mr.min_col, mr.max_col + 1):
                if col_idx not in col_map:
                    col_map[col_idx] = anchor_str
    # Preenche restantes como col_N
    for col_idx in range(1, (ws.max_column or 0) + 1):
        if col_idx not in col_map:
            col_map[col_idx] = f'col_{col_idx}'
    return col_map

def _extract_rows_merged(wb, header_row):
    ws = wb.active
    col_map = _resolve_merged_headers(ws, header_row)
    max_col = ws.max_column or 0
    headers = [col_map.get(i, f'col_{i}') for i in range(1, max_col + 1)]
    rows = []
    for row in ws.iter_rows(min_row=header_row + 1, values_only=True):
        if all(v is None for v in row): continue
        rows.append(dict(zip(headers, row)))
    return rows

header_row   = _detect_header_merged(ws, KEYWORDS_HEADER)
roteiro_rows = _extract_rows_merged(wb, header_row)
print(f'     {len(roteiro_rows)} linhas de teste (cabeçalho na linha {header_row})')

# Diagnóstico: mostra os primeiros 10 headers detectados
if roteiro_rows:
    hdrs = list(roteiro_rows[0].keys())[:10]
    print(f'     Headers detectados: {hdrs}')

# ── Helper: escrita segura em células (mesmo mescladas) ──
def _find_merged_range(ws, row, col):
    from openpyxl.utils import get_column_letter
    coord = f'{get_column_letter(col)}{row}'
    for mr in ws.merged_cells.ranges:
        if coord in mr: return mr
    return None

def _safe_write(ws, row, col, value, fill):
    # Desfaz merge se necessário
    mr = _find_merged_range(ws, row, col)
    if mr:
        anchor = ws.cell(row=mr.min_row, column=mr.min_col)
        sv, sf = anchor.value, anchor.fill
        ws.unmerge_cells(str(mr))
        anchor.value, anchor.fill = sv, sf
    try:
        cell = ws.cell(row=row, column=col)
        cell.value = value
        cell.fill  = fill
    except AttributeError:
        # Fallback: tenta desfazer qualquer merge residual
        mr2 = _find_merged_range(ws, row, col)
        if mr2: ws.unmerge_cells(str(mr2))
        try:
            cell = ws.cell(row=row, column=col)
            cell.value = value
            cell.fill  = fill
        except AttributeError as e:
            print(f'  ⚠️ Não foi possível escrever em ({row},{col}): {e}')

# Monkey-patch do ResultWriter para usar _safe_write desta célula
from src import result_writer as _rw_mod
_rw_mod._safe_write = _safe_write
_rw_mod._find_merged_range = _find_merged_range

runner  = TestRunner(audit_parser, pdf_parser, promo_engine)
writer  = ResultWriter(wb, OUTPUT_XLSX)
alogger = AuditLogger(OUTPUT_LOG)

# ── 6. Loop principal ────────────────────────────────────
print()
print('─' * 55)
print(f'{"Linha":>6}  {"Status":8}  Detalhe')
print('─' * 55)

test_results = []
for idx, row in enumerate(roteiro_rows, start=1):
    data_row = header_row + idx
    result   = runner.run(row, linha=data_row)
    # Escreve diretamente usando _safe_write para garantir compat
    status = 'Ok'    if result.passed else 'Erro'
    fill   = PatternFill('solid', fgColor='C6EFCE') if result.passed else PatternFill('solid', fgColor='FFC7CE')
    for col in (writer.col_sat, writer.col_ecf, writer.col_nfce):
        if col: _safe_write(ws, data_row, col, status, fill)
    if writer.col_just:
        motivo = result.motivo_erro if not result.passed else ''
        _safe_write(ws, data_row, writer.col_just, motivo, PatternFill())
    alogger.record(result)
    test_results.append(result)

    icon   = '🟢' if result.passed else '🔴'
    motivo = f'  ← {result.motivo_erro}' if not result.passed else ''
    print(f'{data_row:>6}  {icon} {"OK" if result.passed else "ERRO":6}  {motivo}')

# ── 7. Persistência ──────────────────────────────────────
wb.save(OUTPUT_XLSX)
alogger.flush()

sumario = alogger.summary()
print()
print('=' * 55)
print(f"✅ Concluído — {sumario['passou']}/{sumario['total']} passaram ({sumario['pct_ok']}%)")
print(f"   🟢 OK    : {sumario['passou']}")
print(f"   🔴 ERRO  : {sumario['falhou']}")
print(f'   📁 XLSX  : {OUTPUT_XLSX}')
print(f'   📋 LOG   : {OUTPUT_LOG}')

---
## 📊 Célula 8 — Resumo detalhado e tabela de resultados

In [ ]:
import pandas as pd, json

with open(OUTPUT_LOG, encoding='utf-8') as f:
    log_entries = json.load(f)

passou = sum(1 for e in log_entries if e['passou'])
falhou = len(log_entries) - passou

print('=' * 60)
print('📊 Etapa 01 — Resumo da Validação')
print('=' * 60)
print(f'Total de testes : {len(log_entries)}')
print(f'🟢 Passou       : {passou}')
print(f'🔴 Falhou       : {falhou}')
print()

records = []
for entry in log_entries:
    base = {
        'etapa':       entry.get('etapa'),
        'linha':       entry.get('linha'),
        'cupom':       entry.get('cupom_numero'),
        'passou':      '🟢 OK' if entry['passou'] else '🔴 ERRO',
        'motivo_erro': entry.get('motivo_erro', ''),
    }
    for ck in entry.get('checks', []):
        base[ck['check']] = '✅' if ck['ok'] else f"❌ {ck['detalhe'][:40]}"
    records.append(base)

df_res = pd.DataFrame(records)

def _highlight(val):
    if isinstance(val, str) and val.startswith('❌'):
        return 'background-color: #FFC7CE; color: #9C0006'
    if isinstance(val, str) and val == '🔴 ERRO':
        return 'background-color: #FFC7CE; font-weight: bold'
    if isinstance(val, str) and val == '🟢 OK':
        return 'background-color: #C6EFCE; color: #276221'
    return ''

display(df_res.style.applymap(_highlight))

---
## 📥 Célula 9 — Download dos resultados

In [ ]:
from google.colab import files as colab_files
import os

for arquivo in [OUTPUT_XLSX, OUTPUT_LOG, 'output/qa_validation.log']:
    if os.path.exists(arquivo):
        colab_files.download(arquivo)
        print(f'📥 Download iniciado: {arquivo}')
    else:
        print(f'⚠️ Arquivo não encontrado: {arquivo}')

---
## 🔍 Célula 10 — Diagnóstico de um teste específico

> Altere `LINHA_INSPECIONAR` para o número da linha do xlsx que quer depurar.

In [ ]:
import json

LINHA_INSPECIONAR = header_row + 1

entry = next((e for e in log_entries if e['linha'] == LINHA_INSPECIONAR), None)

if entry is None:
    print(f'❌ Linha {LINHA_INSPECIONAR} não encontrada no log.')
else:
    print(f'🔍 Diagnóstico — Linha {LINHA_INSPECIONAR} | Etapa: {entry["etapa"]}')
    print(f'   Cupom  : {entry["cupom_numero"]}')
    print(f'   Status : {"🟢 PASSOU" if entry["passou"] else "🔴 FALHOU"}')
    if entry['motivo_erro']:
        print(f'   Motivo : {entry["motivo_erro"]}')
    print()
    print(f'   {"Check":<25} {"Resultado":<8}  Detalhe')
    print('   ' + '─' * 65)
    for ck in entry['checks']:
        icon    = '✅' if ck['ok'] else '❌'
        detalhe = ck['detalhe'] if not ck['ok'] else ''
        print(f'   {icon} {ck["check"]:<24} {detalhe}')
    if entry.get('cupom_numero'):
        movs = audit_parser.get_by_numero(entry['cupom_numero'])
        print()
        if movs:
            print(f'   📋 Audit — {len(movs)} movimento(s) para cupom {entry["cupom_numero"]}:')
            for i, m in enumerate(movs, 1):
                campos = {k: m[k] for k in ['total','descuentoTotal','cancelacion','status','codigoTipoPago'] if k in m}
                print(f'     [{i}] {campos}')
        else:
            print(f'   ⚠️ Cupom {entry["cupom_numero"]} não encontrado no Audit')

---
## 🧪 Célula 11 — Teste Unitário com Mocks (sem arquivos reais)

> Execute esta célula **a qualquer momento** para validar que os módulos estão
> funcionando corretamente, **sem precisar de arquivos reais**.

In [ ]:
import json, os
import pandas as pd
import openpyxl

from src.audit_parser      import AuditParser
from src.coupon_pdf_parser import CouponPDFParser
from src.promo_engine      import PromoEngine
from src.test_runner       import TestRunner
from src.result_writer     import ResultWriter
from src.audit_logger      import AuditLogger

PASS = '🟢 PASS'
FAIL = '🔴 FAIL'
resultados_mock = []

def chk(nome, ok, detalhe=''):
    label = PASS if ok else FAIL
    msg   = f' → {detalhe}' if (detalhe and not ok) else ''
    print(f'  {label}  {nome}{msg}')
    resultados_mock.append((nome, ok))

print('=' * 60)
print('🧪 Testes Unitários — Etapa 01 (Mocks)')
print('=' * 60)

print('\n[M2] AuditParser')
mov_data = {'numero':'001','total':99.90,'descuentoTotal':0.0,'cancelacion':False,'status':200,
            'codigoTipoPago':'1','detalles':[{'ean':'7891234567890','precio':99.90,'descuento':0.0,'participante':True}],'pagos':[]}
json_ok      = json.dumps(mov_data)
json_escaped = json_ok.replace('"','""')
df_mock = pd.DataFrame([{'Request': json_escaped}, {'Request': 'INVALIDO'}])
ap = AuditParser(df_mock)
chk('Indexa cupom válido',              len(ap.get_by_numero('001')) == 1)
chk('Retorna [] para inexistente',      ap.get_by_numero('999') == [])
chk('Busca por EAN funciona',           len(ap.get_by_eans(['7891234567890'])) >= 1)

print('\n[M3] CouponPDFParser')
pdf_mock = ['NFC-e 0042\nSubtotal: 99,90\nDesconto: 10,00\nTotal: 89,90\n7891234567890\nDinheiro',
            'SAT FISCAL 123456\nSubtotal: 50,00\nTotal: 50,00\n7891111111111\nCartão Débito']
pp = CouponPDFParser(pdf_mock)
chk('Extrai 2 cupons',          len(pp.coupons) == 2)
c1 = pp.get_by_numero('0042')
chk('Localiza NFC-e 0042',      c1 is not None)
chk('Total = 89.90',            c1 is not None and abs((c1.total or 0)-89.90) < 0.05)

print('\n[M4] PromoEngine')
pe = PromoEngine()
r = pe.validate('LLEVAPAGA',{'descuentoTotal':10.0},{'qtd':3,'trigger':3,'paga':2,'preco_unit':10.0})
chk('LLEVAPAGA OK',   r.ok)
r = pe.validate_pagamento('1','dinheiro')
chk('Pagamento OK',   r.ok)
r = pe.validate_schema({'total':10,'numero':'001','detalles':[],'pagos':[]})
chk('Schema OK',      r.ok)

print('\n[M5] TestRunner')
runner_mock = TestRunner(ap, pp, pe)
if ap.get_by_numero('001'): ap.get_by_numero('001')[0].update(mov_data)
row_ok = {'etapa':'E01','numero_cupom':'001','eans':['7891234567890'],'total':99.90,'desconto':0.0,
          'meio_pagamento':'dinheiro','observacao':'','bin':'','tipo_promo':''}
res = runner_mock.run(row_ok, linha=2)
chk('Cupom localizado', any(c.check=='cupom_localizado' and c.ok for c in res.checks))
chk('HTTP 200',         any(c.check=='http_200'         and c.ok for c in res.checks))
chk('Schema OK',        any(c.check=='schema_json'      and c.ok for c in res.checks))

print('\n[M6+M7] ResultWriter + AuditLogger')
wb_mock = openpyxl.Workbook()
ws_mock = wb_mock.active
ws_mock.append(['etapa','tipo promo','total','desconto','sat','ecf','nfce','motivo'])
ws_mock.append(['E01','','99.90','0.0','','','',''])
TMP_XLSX = '/tmp/mock_resultado.xlsx'
TMP_LOG  = '/tmp/mock_log.json'
rw = ResultWriter(wb_mock, TMP_XLSX)
rw.write_result(res, data_row=2)
rw.save()
chk('XLSX salvo', os.path.exists(TMP_XLSX))
wb_check = openpyxl.load_workbook(TMP_XLSX)
cell_sat = wb_check.active.cell(row=2, column=rw.col_sat)
chk('Célula SAT preenchida', cell_sat.value in ('Ok','Erro'))
al = AuditLogger(TMP_LOG)
al.record(res); al.flush()
with open(TMP_LOG) as f: log_mock = json.load(f)
chk('AuditLogger: 1 entrada', len(log_mock) == 1)
chk('Campo checks presente',  'checks' in log_mock[0])

total_t = len(resultados_mock)
total_p = sum(1 for _,ok in resultados_mock if ok)
print()
print('=' * 60)
print(f'🏁 {total_p}/{total_t} passaram ({round(total_p/total_t*100,1)}%)')
if total_t - total_p:
    print('   ⚠️  Falhas — revise os módulos.')
else:
    print('   ✅ Todos os módulos validados!')